<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-30.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 30 - Consultas avanzadas Pig y Hive

**Módulo:** Big Data Aplicado (BDA - 5075)

---

## Objetivos

- Simular consultas avanzadas de Pig Latin (COGROUP, operaciones anidadas)
- Implementar **funciones de ventana** (window functions) con Pandas y SQLite
- Comparar rendimiento con y sin optimizaciones
- Practicar patrones frecuentes: ranking, acumulados, comparaciones temporales

## 1. Preparación de datos

In [ ]:
import pandas as pd
import sqlite3
import random
import time

random.seed(42)

categorias = ['Electrónica', 'Ropa', 'Hogar', 'Deportes', 'Libros']
ciudades = ['Madrid', 'Barcelona', 'Valencia', 'Sevilla', 'Bilbao']
productos_cat = {
    'Electrónica': ['Portátil', 'Móvil', 'Tablet', 'Auriculares', 'Monitor'],
    'Ropa': ['Camiseta', 'Pantalón', 'Chaqueta', 'Zapatos', 'Bufanda'],
    'Hogar': ['Lámpara', 'Silla', 'Mesa', 'Alfombra', 'Cortinas'],
    'Deportes': ['Balón', 'Raqueta', 'Bicicleta', 'Mancuernas', 'Esterilla'],
    'Libros': ['Novela', 'Manual técnico', 'Cómic', 'Biografía', 'Ensayo']
}

# Generar ventas
ventas_data = []
for i in range(2000):
    cat = random.choice(categorias)
    mes = random.randint(1, 12)
    dia = random.randint(1, 28)
    ventas_data.append({
        'id': i + 1,
        'producto': random.choice(productos_cat[cat]),
        'categoria': cat,
        'importe': round(random.uniform(10, 500), 2),
        'ciudad': random.choice(ciudades),
        'cliente_id': random.randint(1, 200),
        'fecha': f'2025-{mes:02d}-{dia:02d}',
        'mes': mes,
        'trimestre': f'Q{(mes-1)//3+1}'
    })

ventas = pd.DataFrame(ventas_data)

# Generar devoluciones (20% de las ventas)
devoluciones_data = []
motivos = ['Defectuoso', 'No coincide', 'Cambio de opinión', 'Talla incorrecta', 'Llegó tarde']
for i, venta in ventas.sample(frac=0.2, random_state=42).iterrows():
    devoluciones_data.append({
        'devolucion_id': len(devoluciones_data) + 1,
        'venta_id': venta['id'],
        'producto': venta['producto'],
        'categoria': venta['categoria'],
        'importe': venta['importe'],
        'motivo': random.choice(motivos)
    })

devoluciones = pd.DataFrame(devoluciones_data)

# Crear conexión SQLite
conn = sqlite3.connect(':memory:')
ventas.to_sql('ventas', conn, index=False, if_exists='replace')
devoluciones.to_sql('devoluciones', conn, index=False, if_exists='replace')

print(f"Ventas: {len(ventas)} registros")
print(f"Devoluciones: {len(devoluciones)} registros")
print(f"\nVentas (muestra):")
print(ventas.head().to_string(index=False))
print(f"\nDevoluciones (muestra):")
print(devoluciones.head().to_string(index=False))

## 2. Pig Latin avanzado: COGROUP simulado

COGROUP agrupa dos relaciones por una clave común, manteniendo los datos de cada relación en bags separados.

In [ ]:
# === Simulación de COGROUP en Pig Latin ===
# Pig: cogrupado = COGROUP ventas BY producto, devoluciones BY producto;

print("=== PIG: COGROUP ventas BY producto, devoluciones BY producto ===")
print()

# En Pandas: agrupamos ambas relaciones por la misma clave
ventas_por_prod = ventas.groupby('producto')
devol_por_prod = devoluciones.groupby('producto')

todos_productos = set(ventas['producto'].unique()) | set(devoluciones['producto'].unique())

cogroup_resultado = []
for prod in sorted(todos_productos):
    v_bag = ventas_por_prod.get_group(prod) if prod in ventas_por_prod.groups else pd.DataFrame()
    d_bag = devol_por_prod.get_group(prod) if prod in devol_por_prod.groups else pd.DataFrame()
    cogroup_resultado.append({
        'producto': prod,
        'num_ventas': len(v_bag),
        'total_ventas': round(v_bag['importe'].sum(), 2) if len(v_bag) > 0 else 0,
        'num_devoluciones': len(d_bag),
        'total_devuelto': round(d_bag['importe'].sum(), 2) if len(d_bag) > 0 else 0
    })

cogroup_df = pd.DataFrame(cogroup_resultado)
cogroup_df['tasa_devolucion'] = round(cogroup_df['num_devoluciones'] / cogroup_df['num_ventas'] * 100, 1)
cogroup_df['beneficio_neto'] = round(cogroup_df['total_ventas'] - cogroup_df['total_devuelto'], 2)

print(cogroup_df.sort_values('tasa_devolucion', ascending=False).to_string(index=False))

print("\n--- Productos con mayor tasa de devolución ---")
print(cogroup_df.nlargest(5, 'tasa_devolucion')[['producto', 'num_ventas', 'num_devoluciones', 'tasa_devolucion']].to_string(index=False))

In [ ]:
# === Simulación de FOREACH anidado en Pig ===
# Pig:
# agrupado = GROUP ventas BY categoria;
# stats = FOREACH agrupado {
#     ordenadas = ORDER ventas BY importe DESC;
#     top3 = LIMIT ordenadas 3;
#     grandes = FILTER ventas BY importe > 200;
#     GENERATE group, COUNT(ventas), COUNT(grandes), FLATTEN(top3);
# };

print("=== PIG: FOREACH anidado con operaciones en bags ===")
print()

for cat, grupo in ventas.groupby('categoria'):
    # Operaciones anidadas dentro del FOREACH
    ordenadas = grupo.sort_values('importe', ascending=False)
    top3 = ordenadas.head(3)
    grandes = grupo[grupo['importe'] > 200]

    print(f"Categoría: {cat}")
    print(f"  Total ventas: {len(grupo)}")
    print(f"  Ventas > 200€: {len(grandes)}")
    print(f"  Top 3 por importe:")
    for _, row in top3.iterrows():
        print(f"    {row['producto']:20s} {row['importe']:8.2f}€")
    print()

## 3. Window Functions con Pandas (simulando HiveQL)

### 3.1 Funciones de ranking

In [ ]:
# === ROW_NUMBER() OVER (PARTITION BY categoria ORDER BY importe DESC) ===
print("=== HIVEQL: ROW_NUMBER() — Top 3 productos por categoría ===")
print()

# Agrupar ventas por producto y categoría
prod_totales = ventas.groupby(['categoria', 'producto']).agg(
    total_ventas=('importe', 'sum'),
    num_ventas=('id', 'count')
).reset_index()

# ROW_NUMBER simulado
prod_totales['row_number'] = prod_totales.groupby('categoria')['total_ventas'].rank(
    method='first', ascending=False
).astype(int)

# Top 3 por categoría
top3 = prod_totales[prod_totales['row_number'] <= 3].sort_values(
    ['categoria', 'row_number']
)

print("Equivalente HiveQL:")
print("  SELECT * FROM (")
print("    SELECT categoria, producto, SUM(importe) AS total,")
print("      ROW_NUMBER() OVER (PARTITION BY categoria ORDER BY SUM(importe) DESC) AS rn")
print("    FROM ventas GROUP BY categoria, producto")
print("  ) ranked WHERE rn <= 3;")
print()
print(top3[['categoria', 'producto', 'total_ventas', 'num_ventas', 'row_number']].to_string(index=False))

In [ ]:
# === Window Functions: SUM acumulado, LAG, LEAD ===
print("=== HIVEQL: Funciones de ventana — análisis temporal ===\n")

# Agregar ventas por mes
ventas_mensuales = ventas.groupby('mes').agg(
    num_ventas=('id', 'count'),
    total=('importe', 'sum')
).reset_index().sort_values('mes')
ventas_mensuales['total'] = ventas_mensuales['total'].round(2)

# SUM() OVER (ORDER BY mes ROWS UNBOUNDED PRECEDING) → acumulado
ventas_mensuales['acumulado'] = ventas_mensuales['total'].cumsum().round(2)

# LAG(total, 1) OVER (ORDER BY mes) → mes anterior
ventas_mensuales['mes_anterior'] = ventas_mensuales['total'].shift(1)

# Variación respecto al mes anterior
ventas_mensuales['variacion_pct'] = (
    (ventas_mensuales['total'] - ventas_mensuales['mes_anterior'])
    / ventas_mensuales['mes_anterior'] * 100
).round(1)

# AVG() OVER (ORDER BY mes ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) → media móvil
ventas_mensuales['media_movil_3m'] = ventas_mensuales['total'].rolling(window=3).mean().round(2)

# LEAD(total, 1) OVER (ORDER BY mes) → mes siguiente
ventas_mensuales['mes_siguiente'] = ventas_mensuales['total'].shift(-1)

print("Equivalente HiveQL:")
print("  SELECT mes, total,")
print("    SUM(total) OVER (ORDER BY mes ROWS UNBOUNDED PRECEDING) AS acumulado,")
print("    LAG(total, 1) OVER (ORDER BY mes) AS mes_anterior,")
print("    AVG(total) OVER (ORDER BY mes ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS media_movil")
print("  FROM ventas_mensuales;")
print()
print(ventas_mensuales.to_string(index=False))

In [ ]:
# === Window Functions: RANK y NTILE ===
print("=== HIVEQL: RANK() y NTILE() ===\n")

# RANK() vs DENSE_RANK() vs ROW_NUMBER()
ciudad_totales = ventas.groupby('ciudad')['importe'].sum().reset_index()
ciudad_totales.columns = ['ciudad', 'total']
ciudad_totales['total'] = ciudad_totales['total'].round(2)
ciudad_totales = ciudad_totales.sort_values('total', ascending=False)

ciudad_totales['row_number'] = range(1, len(ciudad_totales) + 1)
ciudad_totales['rank'] = ciudad_totales['total'].rank(method='min', ascending=False).astype(int)
ciudad_totales['dense_rank'] = ciudad_totales['total'].rank(method='dense', ascending=False).astype(int)

print("Comparación ROW_NUMBER vs RANK vs DENSE_RANK:")
print(ciudad_totales.to_string(index=False))

# NTILE: dividir en grupos iguales
print("\n--- NTILE(4): dividir clientes en cuartiles por gasto ---")
cliente_gasto = ventas.groupby('cliente_id')['importe'].sum().reset_index()
cliente_gasto.columns = ['cliente_id', 'gasto_total']
cliente_gasto['gasto_total'] = cliente_gasto['gasto_total'].round(2)
cliente_gasto = cliente_gasto.sort_values('gasto_total', ascending=False)
cliente_gasto['ntile_4'] = pd.qcut(cliente_gasto['gasto_total'], q=4, labels=['Q4-Bajo', 'Q3', 'Q2', 'Q1-Alto'])

resumen_ntile = cliente_gasto.groupby('ntile_4').agg(
    clientes=('cliente_id', 'count'),
    gasto_min=('gasto_total', 'min'),
    gasto_max=('gasto_total', 'max'),
    gasto_medio=('gasto_total', 'mean')
).round(2)
print(resumen_ntile.to_string())

## 4. Window Functions con SQLite (HiveQL real)

SQLite 3.25+ soporta funciones de ventana. Ejecutamos consultas HiveQL reales.

In [ ]:
# === HiveQL con funciones de ventana en SQLite ===

print("=== CONSULTA 1: Ranking por categoría ===")
q1 = """
SELECT * FROM (
    SELECT
        categoria,
        producto,
        ROUND(SUM(importe), 2) AS total,
        COUNT(*) AS ventas,
        ROW_NUMBER() OVER (
            PARTITION BY categoria
            ORDER BY SUM(importe) DESC
        ) AS ranking
    FROM ventas
    GROUP BY categoria, producto
) ranked
WHERE ranking <= 3
ORDER BY categoria, ranking;
"""
print(pd.read_sql(q1, conn).to_string(index=False))

print("\n=== CONSULTA 2: Acumulado mensual con variación ===")
q2 = """
SELECT
    mes,
    num_ventas,
    total,
    SUM(total) OVER (ORDER BY mes ROWS UNBOUNDED PRECEDING) AS acumulado,
    LAG(total, 1) OVER (ORDER BY mes) AS mes_anterior,
    ROUND(
        (total - LAG(total, 1) OVER (ORDER BY mes))
        * 100.0 / LAG(total, 1) OVER (ORDER BY mes), 1
    ) AS variacion_pct
FROM (
    SELECT
        mes,
        COUNT(*) AS num_ventas,
        ROUND(SUM(importe), 2) AS total
    FROM ventas
    GROUP BY mes
)
ORDER BY mes;
"""
print(pd.read_sql(q2, conn).to_string(index=False))

In [ ]:
print("=== CONSULTA 3: Media móvil de 3 meses por categoría ===")
q3 = """
SELECT
    categoria,
    mes,
    ROUND(total, 2) AS total,
    ROUND(AVG(total) OVER (
        PARTITION BY categoria
        ORDER BY mes
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2) AS media_movil_3m
FROM (
    SELECT categoria, mes, SUM(importe) AS total
    FROM ventas
    GROUP BY categoria, mes
)
ORDER BY categoria, mes;
"""
resultado_q3 = pd.read_sql(q3, conn)
# Mostrar solo las primeras filas de cada categoría
for cat in resultado_q3['categoria'].unique():
    print(f"\n  {cat}:")
    subset = resultado_q3[resultado_q3['categoria'] == cat]
    print(f"  {subset.to_string(index=False)}")

print("\n=== CONSULTA 4: Porcentaje sobre el total de su categoría ===")
q4 = """
SELECT
    producto,
    categoria,
    ROUND(SUM(importe), 2) AS total_producto,
    ROUND(SUM(SUM(importe)) OVER (PARTITION BY categoria), 2) AS total_categoria,
    ROUND(
        SUM(importe) * 100.0 / SUM(SUM(importe)) OVER (PARTITION BY categoria), 1
    ) AS pct_categoria
FROM ventas
GROUP BY producto, categoria
ORDER BY categoria, pct_categoria DESC;
"""
print(pd.read_sql(q4, conn).to_string(index=False))

## 5. Simulación de impacto de optimizaciones

Comparamos el rendimiento con y sin "optimizaciones" simuladas (particionado, formato columnar).

In [ ]:
import os

# Generar dataset más grande para medir diferencias
n_grande = 100000
ventas_grandes = []
for i in range(n_grande):
    cat = random.choice(categorias)
    mes = random.randint(1, 12)
    ventas_grandes.append({
        'id': i+1, 'producto': random.choice(productos_cat[cat]),
        'categoria': cat, 'importe': round(random.uniform(10, 500), 2),
        'ciudad': random.choice(ciudades), 'mes': mes,
        'trimestre': f'Q{(mes-1)//3+1}'
    })
df_grande = pd.DataFrame(ventas_grandes)

print(f"Dataset generado: {len(df_grande)} registros")

# === Consulta SIN optimización: escaneo completo ===
t0 = time.time()
resultado_sin = df_grande[df_grande['mes'] == 6].groupby('categoria')['importe'].agg(['sum', 'mean', 'count'])
t_sin = (time.time() - t0) * 1000

# === Consulta CON particionado simulado ===
particiones = {mes: grupo for mes, grupo in df_grande.groupby('mes')}

t0 = time.time()
resultado_con = particiones[6].groupby('categoria')['importe'].agg(['sum', 'mean', 'count'])
t_con = (time.time() - t0) * 1000

print(f"\n=== Comparativa de rendimiento ===")
print(f"Sin partición (escaneo de {len(df_grande)} filas): {t_sin:.2f} ms")
print(f"Con partición (escaneo de {len(particiones[6])} filas): {t_con:.2f} ms")
print(f"Datos leídos sin partición: 100%")
print(f"Datos leídos con partición: {len(particiones[6])/len(df_grande)*100:.1f}%")

# === Simulación: formato columnar vs filas ===
print(f"\n=== Simulación: Formato columnar vs filas ===")

# "Filas": leer todo y luego seleccionar columna
t0 = time.time()
for _ in range(100):
    _ = df_grande[['id', 'producto', 'categoria', 'importe', 'ciudad', 'mes', 'trimestre']]['importe'].sum()
t_filas = (time.time() - t0) * 1000 / 100

# "Columnar": leer solo la columna necesaria
col_importe = df_grande['importe']
t0 = time.time()
for _ in range(100):
    _ = col_importe.sum()
t_columnar = (time.time() - t0) * 1000 / 100

print(f"Formato filas (lee todas las columnas): {t_filas:.3f} ms/consulta")
print(f"Formato columnar (lee solo 'importe'):  {t_columnar:.3f} ms/consulta")
print(f"\nEn Hive real, ORC/Parquet puede ser 5-10x más rápido que TextFile")

## 6. Ejercicios propuestos

### Ejercicio 1: Window Functions
Escribe una consulta HiveQL (SQLite) que muestre, para cada venta:
- El ranking de esa venta dentro de su ciudad (por importe)
- El importe acumulado en esa ciudad
- La diferencia con la venta anterior en esa ciudad

### Ejercicio 2: COGROUP avanzado
Simula un COGROUP de 3 relaciones: ventas, devoluciones y un nuevo dataset de "reclamaciones". Calcula por producto: ventas netas, tasa de devolución y tasa de reclamación.

### Ejercicio 3: Diseño de optimización
Dado un escenario con una tabla de 500 millones de registros de transacciones bancarias (fecha, cuenta, tipo, importe, sucursal), propón:
- Estrategia de particionado
- Formato de almacenamiento
- Bucketing (si aplica)
- Configuraciones de Hive recomendadas

Justifica cada decisión.

In [ ]:
# === EJERCICIO 1: Window Functions ===
# Tu consulta SQL aquí:
# q_ej1 = """
# SELECT ciudad, producto, importe,
#     ROW_NUMBER() OVER (PARTITION BY ciudad ORDER BY importe DESC) AS rank_ciudad,
#     SUM(importe) OVER (PARTITION BY ciudad ORDER BY id ROWS UNBOUNDED PRECEDING) AS acumulado,
#     importe - LAG(importe, 1) OVER (PARTITION BY ciudad ORDER BY id) AS diff_anterior
# FROM ventas
# LIMIT 20;
# """
# print(pd.read_sql(q_ej1, conn).to_string(index=False))


# === EJERCICIO 2: COGROUP de 3 relaciones ===
# Tu código aquí:


# === EJERCICIO 3: Diseño de optimización ===
# Tu respuesta aquí (como print statements o comentarios):


print("Completa los ejercicios arriba.")

## Recursos

- [Hive Window Functions](https://cwiki.apache.org/confluence/display/Hive/LanguageManual+WindowingAndAnalytics)
- [SQL Window Functions Tutorial](https://mode.com/sql-tutorial/sql-window-functions/)
- [Hive Performance Tuning](https://cwiki.apache.org/confluence/display/Hive/PerformanceTuning)
- [Pig Latin Reference](https://pig.apache.org/docs/latest/basic.html)

---

[← Sesión 29](./sesion-29.ipynb) | [Índice](../sesiones/index.md) | [Siguiente sesión →](./sesion-31.ipynb)